# 04a v13 — ECG Forecasting: CNN-BiLSTM U-Net with Focal-Frequency Loss

## What was wrong with v12 (and why loss stuck at 7.x)

| Problem | Evidence | Fix |
|---|---|---|
| **AdaptiveAvgPool1d(1) killed temporal info** | Compressing 490 timesteps → 1 vector loses all QRS timing. Decoder then interpolates from a single mean embedding — structurally impossible to reconstruct sharp spikes | Pool to **32** (8x downsampling), not 1 |
| **No skip connections** | Encoder fine details (QRS rise/fall at 5-40 Hz) had no path to the decoder; decoder had to hallucinate them from the bottleneck | **U-Net skip connections** from every encoder stage to matching decoder stage |
| **No temporal memory in bottleneck** | Pure MLP bottleneck can't model periodic heart rate (RR interval) which determines WHERE next QRS will be | **BiLSTM** over the 32-step bottleneck sequence captures periodicity |
| **FFT loss weighted uniformly** | Baseline wander (0-0.5 Hz) has large magnitude and dominated the FFT loss; QRS content (5-40 Hz) was underweighted | **Focal-frequency loss** — upweights high-frequency bins 5× |
| **MAE for sharp spikes** | L1 on normalized QRS spikes (amplitude 3-5 sigma) still provides weak gradient signal | **Huber loss** (delta=0.5) — L2 near zero (strong gradient for small errors), L1 for outliers (won't be dominated by missed QRS) |
| **LR too high too fast** | OneCycle 15% warmup with 2.6M param model — gradient explosion in first epochs pushed into bad basin | Cosine annealing with **warm restarts** (T0=20 epochs) + slower warmup |

## Architecture: CNN-BiLSTM U-Net
```
Input (B, 12, 490)
   │
   ▼
[EncBlock1] ──────────────────────────────────── skip1 (B,64,490)
   │ stride 2
   ▼
[EncBlock2] ──────────────────────────────── skip2 (B,128,245)
   │ stride 2
   ▼
[EncBlock3] ──────────────────────────── skip3 (B,256,123)
   │ stride 2
   ▼
[EncBlock4] ──────────────────────────── skip4 (B,256,62)
   │ stride 2
   ▼
(B,256,31) → BiLSTM(256) → (B,31,256) → back to (B,256,31)
   │
   ▼
[DecBlock4] ← cat(skip4) → upsample → (B,256,62)
   │
   ▼
[DecBlock3] ← cat(skip3) → upsample → (B,128,124)
   │
   ▼
[DecBlock2] ← cat(skip2) → upsample → (B,64,248)
   │
   ▼
[DecBlock1] ← cat(skip1) → upsample → (B,32,496)
   │ crop to 490
   ▼
Conv1d(32,12,1) → (B,12,490) → permute → (B,490,12)
```


In [ ]:
# CELL 1 — IMPORTS
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.signal import find_peaks
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('OK')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 — LOAD + NORMALISE
# Global z-score only (PTB-XL is already clipped to [-3,3] mV; IQR≈1.0 so
# per-lead IQR is a no-op and was removed in v12).
PROJECT_ROOT = '/content/drive/MyDrive/his/HIS Project'
if os.path.exists(PROJECT_ROOT):
    os.chdir(PROJECT_ROOT)
    SAVE_DIR = 'data/processed'
    FIG_DIR  = 'reports/figures/cnn_v13'
    CKPT_DIR = 'checkpoints'
else:
    SAVE_DIR = os.path.join('..', 'data', 'processed')
    FIG_DIR  = os.path.join('..', 'reports', 'figures', 'cnn_v13')
    CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')

os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'CWD  : {os.getcwd()}')
print(f'Data : {SAVE_DIR}')

X_train = np.load(os.path.join(SAVE_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(SAVE_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

LEAD_NAMES = cfg['lead_names']
FS         = cfg['sampling_rate']
INPUT_LEN  = cfg['input_len']
HORIZON    = cfg['horizon']
N_LEADS    = cfg['n_leads']

print(f'X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'y_train  mean={y_train.mean():.4f}  std={y_train.std():.4f}  '
      f'min={y_train.min():.3f}  max={y_train.max():.3f}')

# --- z-score normalisation (fit on train only) ---
def fit_norm(X, y):
    vals = np.concatenate([X.ravel(), y.ravel()])
    mu, sigma = float(vals.mean()), float(vals.std())
    return mu, max(sigma, 1e-6)

NORM_MU, NORM_SIGMA = fit_norm(X_train, y_train)

def normalize(arr):   return (arr - NORM_MU) / NORM_SIGMA
def denormalize(arr): return arr * NORM_SIGMA + NORM_MU

X_train = normalize(X_train)
y_train = normalize(y_train)
X_val   = normalize(X_val)
y_val   = normalize(y_val)
X_test  = normalize(X_test)
y_test  = normalize(y_test)

print(f'Norm: mu={NORM_MU:.5f}  sigma={NORM_SIGMA:.5f}')
print(f'Normalised y_train: mean={y_train.mean():.4f} std={y_train.std():.4f}')
print('OK Data ready')


In [ ]:
# CELL 3 — DATASET
# Augmentation: ±5% amplitude jitter + random polarity flip (±1).
# No noise injection — destroys QRS morphology.
# No time-warping — changes RR intervals, confuses beat-timing prediction.

class ECGForecastDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X       = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y       = torch.from_numpy(np.asarray(y, dtype=np.float32))
        self.augment = augment

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].clone()   # (T_in, leads)
        y = self.y[idx].clone()   # (T_out, leads)
        if self.augment:
            # ±5% amplitude jitter
            scale = torch.empty(1).uniform_(0.95, 1.05)
            x, y  = x * scale, y * scale
            # Random polarity flip (simulates lead reversal; ECG is linear)
            if torch.rand(1).item() < 0.1:
                x, y = -x, -y
        x = x.permute(1, 0)   # (leads, T_in) for encoder
        return x, y            # y: (T_out, leads)


def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te,
                 batch_train=48, batch_eval=96):
    kw = dict(num_workers=2, pin_memory=(DEVICE.type == 'cuda'),
              persistent_workers=True)
    tr = DataLoader(ECGForecastDataset(X_tr, y_tr, augment=True),
                    batch_size=batch_train, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(ECGForecastDataset(X_v,  y_v,  augment=False),
                    batch_size=batch_eval,  shuffle=False, **kw)
    te = DataLoader(ECGForecastDataset(X_te, y_te, augment=False),
                    batch_size=batch_eval,  shuffle=False, **kw)
    return tr, vl, te


BATCH_TRAIN = 48
cnn_tr, cnn_vl, cnn_te = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test,
    batch_train=BATCH_TRAIN)

xb, yb = next(iter(cnn_tr))
print(f'Train batch : x={tuple(xb.shape)}  y={tuple(yb.shape)}')
print(f'Batches     : train={len(cnn_tr)} | val={len(cnn_vl)} | test={len(cnn_te)}')
print('OK DataLoaders ready  (batch=48, ±5% jitter + 10% polarity flip)')


In [ ]:
# CELL 4 — MODEL: CNN-BiLSTM U-Net
#
# U-Net skip connections are the KEY fix vs v12:
# - Encoder compresses spatial resolution (stride-2 conv) but accumulates channels
# - Skip connections pass FULL-RESOLUTION features to matching decoder stages
# - Decoder only needs to REFINE (add high-freq detail) not hallucinate from scratch
# - This is exactly what CED-U-Net achieves on PTB-XL delineation (literature above)
#
# BiLSTM in the bottleneck:
# - Sees the 32-step compressed sequence (490/16 ≈ 31 steps)
# - Each step corresponds to ~16 samples = ~32ms at 500Hz (sub-beat resolution)
# - BiLSTM learns periodic structure (RR interval) -> tells decoder WHERE next QRS lands
# - Bidirectional: uses context from both ends of the encoded window

class ConvBNGELU(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, d=1):
        super().__init__()
        pad = ((k - 1) * d) // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_c, out_c, k, stride=s, dilation=d, padding=pad, bias=False),
            nn.GroupNorm(min(8, out_c), out_c),
            nn.GELU()
        )
    def forward(self, x): return self.net(x)


class EncoderBlock(nn.Module):
    """Two dilated conv layers, then stride-2 downsample."""
    def __init__(self, in_c, out_c, dilation=1):
        super().__init__()
        self.conv1 = ConvBNGELU(in_c,  out_c, k=5, d=dilation)
        self.conv2 = ConvBNGELU(out_c, out_c, k=5, d=dilation)
        self.down  = nn.Conv1d(out_c, out_c, kernel_size=2, stride=2, bias=False)
        self.drop  = nn.Dropout(0.10)

    def forward(self, x):
        x = self.conv2(self.conv1(x))   # full-res features (will be skip)
        skip = x
        return self.drop(self.down(x)), skip


class DecoderBlock(nn.Module):
    """Upsample, cat skip, two conv layers."""
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up    = nn.ConvTranspose1d(in_c, in_c, kernel_size=2, stride=2)
        self.conv1 = ConvBNGELU(in_c + skip_c, out_c, k=5)
        self.conv2 = ConvBNGELU(out_c,          out_c, k=5)
        self.drop  = nn.Dropout(0.10)

    def forward(self, x, skip):
        x = self.up(x)                          # upsample
        # Trim/pad to match skip size (handles odd lengths after stride-2)
        if x.shape[-1] != skip.shape[-1]:
            x = x[..., :skip.shape[-1]]
        x = torch.cat([x, skip], dim=1)         # channel-cat skip
        return self.drop(self.conv2(self.conv1(x)))


class ECGUNet(nn.Module):
    """
    CNN-BiLSTM U-Net for ECG seq2seq forecasting.
    Input  : (B, n_leads, T_in)  e.g. (B,12,490)
    Output : (B, T_out, n_leads)  e.g. (B,490,12)
    """
    def __init__(self, n_leads=12, horizon=490, dropout=0.10):
        super().__init__()
        self.horizon = horizon

        # ── ENCODER (4 stages, each halves T) ────────────────────────────
        # 490 → 245 → 123 → 62 → 31
        self.enc1 = EncoderBlock( n_leads,  64, dilation=1)   # skip: (B, 64,490)
        self.enc2 = EncoderBlock(      64, 128, dilation=2)   # skip: (B,128,245)
        self.enc3 = EncoderBlock(     128, 256, dilation=4)   # skip: (B,256,123)
        self.enc4 = EncoderBlock(     256, 256, dilation=8)   # skip: (B,256, 62)
        # After enc4 pool: (B,256,31)

        # ── BOTTLENECK: BiLSTM over compressed time axis ──────────────────
        # (B,256,31) -> transpose -> (B,31,256) -> BiLSTM -> (B,31,256) -> back
        LSTM_HIDDEN = 128   # bidirectional -> 256 total
        self.bottleneck_lstm = nn.LSTM(
            input_size=256, hidden_size=LSTM_HIDDEN,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=dropout)
        self.bottleneck_proj = ConvBNGELU(256, 256, k=1)   # fuse after LSTM

        # ── DECODER (4 stages, each doubles T) ───────────────────────────
        # 31 -> 62 -> 124 -> 248 -> 496, then crop to horizon
        self.dec4 = DecoderBlock(256, 256, 256)  # up from 31 + skip4(62)
        self.dec3 = DecoderBlock(256, 256, 128)  # up from 62 + skip3(123)
        self.dec2 = DecoderBlock(128, 128,  64)  # up from 123 + skip2(245)
        self.dec1 = DecoderBlock( 64,  64,  32)  # up from 245 + skip1(490)

        # ── OUTPUT HEAD ──────────────────────────────────────────────────
        self.out_conv = nn.Conv1d(32, n_leads, 1)

    def forward(self, x):
        # x: (B, leads, T_in)
        x, s1 = self.enc1(x)   # x: (B,64,245)   s1: (B,64,490)
        x, s2 = self.enc2(x)   # x: (B,128,123)  s2: (B,128,245)
        x, s3 = self.enc3(x)   # x: (B,256,62)   s3: (B,256,123)
        x, s4 = self.enc4(x)   # x: (B,256,31)   s4: (B,256,62)

        # BiLSTM bottleneck
        B, C, L = x.shape
        lstm_in  = x.permute(0, 2, 1)           # (B,31,256)
        lstm_out, _ = self.bottleneck_lstm(lstm_in)  # (B,31,256)
        x = lstm_out.permute(0, 2, 1)           # (B,256,31)
        x = self.bottleneck_proj(x)

        # Decode
        x = self.dec4(x, s4)   # (B,256,62)
        x = self.dec3(x, s3)   # (B,128,124)
        x = self.dec2(x, s2)   # (B,64,248)
        x = self.dec1(x, s1)   # (B,32,496)

        # Crop/pad to exact horizon length
        x = x[..., :self.horizon]               # (B,32,490)
        x = self.out_conv(x)                    # (B,12,490)
        return x.permute(0, 2, 1)               # (B,490,12)


# Shape verification
_m = ECGUNet(N_LEADS, HORIZON).to(DEVICE)
with torch.no_grad():
    _dummy = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    _out   = _m(_dummy)
    assert _out.shape == (4, HORIZON, N_LEADS), f'Shape mismatch: {_out.shape}'
    _out_std = _out.std().item()
    print(f'Forward pass OK : {tuple(_dummy.shape)} -> {tuple(_out.shape)}')
    print(f'Output std at random init: {_out_std:.4f}  (want > 0.05)')
del _m
print('OK ECGUNet defined (CNN-BiLSTM U-Net)')


In [ ]:
# CELL 5 — INSTANTIATE + PARAM COUNT
model    = ECGUNet(n_leads=N_LEADS, horizon=HORIZON, dropout=0.10).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

with torch.no_grad():
    dummy = torch.randn(2, N_LEADS, INPUT_LEN).to(DEVICE)
    out   = model(dummy)
    print(f'Forward : {tuple(dummy.shape)} -> {tuple(out.shape)}')
    print(f'Output std (random init): {out.std().item():.4f}')

print(f'Parameters : {n_params:,}')
print(f'Device     : {DEVICE}')
print('OK Model ready')


In [ ]:
# CELL 6 — COMPOSITE LOSS: Huber + Gradient + Focal-Frequency
#
# Three terms, each targeting a different failure mode:
#
# 1. Huber (smooth L1, delta=0.5)
#    delta=0.5 means: L2 for |error| < 0.5 (strong gradient near 0),
#    L1 for |error| > 0.5 (robust to large QRS amplitude errors).
#    Better than pure MAE (too weak gradient for small errors on normalised data)
#    and L2 (dominated by QRS spike amplitude, rewards flatness).
#
# 2. Gradient L1 (first difference): flat prediction has zero gradient
#    everywhere; real ECG has sharp QRS transitions up to 10 mV/sample.
#    GRAD_WEIGHT=3.0 (reduced from 4.0 to balance with FFT term).
#
# 3. Focal Frequency Loss: L1 on FFT magnitudes, but HIGH-FREQUENCY bins
#    (> cutoff_hz) are weighted 5x more than low-freq.
#    QRS complex is 5–40 Hz. Baseline wander is 0–0.5 Hz.
#    Without focal weighting, the large low-freq magnitudes dominate FFT loss
#    and the model learns baseline but ignores QRS.

GRAD_WEIGHT = 3.0
FFT_WEIGHT  = 1.0
HF_BOOST    = 5.0    # how much to upweight high-freq bins in FFT loss
HF_CUTOFF   = 5.0    # Hz — QRS starts here
HUBER_DELTA = 0.5

class FocalFrequencyLoss(nn.Module):
    """FFT magnitude L1 with high-frequency upweighting."""
    def __init__(self, fs, horizon, hf_cutoff_hz=HF_CUTOFF, hf_boost=HF_BOOST):
        super().__init__()
        # Pre-compute per-bin weights (scalar tensor, fixed)
        n_bins = horizon // 2 + 1
        freqs  = torch.linspace(0, fs / 2, n_bins)
        w      = torch.ones(n_bins)
        w[freqs >= hf_cutoff_hz] = hf_boost
        # Register as buffer so it moves to GPU automatically
        self.register_buffer('w', w.view(1, 1, -1))  # (1,1,n_bins) for broadcast

    def forward(self, pred, target):
        # pred, target: (B, T, leads) — operate on time axis
        p_fft = torch.fft.rfft(pred.float(),   dim=1).abs()   # (B, n_bins, leads)
        t_fft = torch.fft.rfft(target.float(), dim=1).abs()
        p_fft = p_fft.permute(0, 2, 1)   # (B, leads, n_bins) for weight broadcast
        t_fft = t_fft.permute(0, 2, 1)
        w = self.w.to(pred.device)
        return (w * (p_fft - t_fft).abs()).mean()


class CompositeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ffl = FocalFrequencyLoss(FS, HORIZON)

    def forward(self, pred, target):
        # 1. Huber
        huber = F.huber_loss(pred, target, delta=HUBER_DELTA)
        # 2. Gradient
        pred_d   = pred[:, 1:, :]   - pred[:, :-1, :]
        target_d = target[:, 1:, :] - target[:, :-1, :]
        grad     = F.l1_loss(pred_d, target_d)
        # 3. Focal-frequency
        fft      = self.ffl(pred, target)
        return huber + GRAD_WEIGHT * grad + FFT_WEIGHT * fft


criterion = CompositeLoss().to(DEVICE)

# Sanity check: flat output must be clearly worse than perfect
_sample_t  = torch.from_numpy(y_train[:128].astype('float32'))
_flat_pred = _sample_t.mean(dim=1, keepdim=True).expand_as(_sample_t)
_flat_loss    = criterion(_flat_pred.to(DEVICE), _sample_t.to(DEVICE)).item()
_perfect_loss = criterion(_sample_t.to(DEVICE), _sample_t.to(DEVICE)).item()
_gap          = _flat_loss - _perfect_loss
print(f'Flat prediction loss   : {_flat_loss:.4f}')
print(f'Perfect prediction loss: {_perfect_loss:.4f}')
print(f'Gap (flat vs perfect)  : {_gap:.4f}')
if _gap < 0.5:
    print('WARNING: gap too small — raise GRAD_WEIGHT or HF_BOOST')
else:
    print('OK Flat predictions strongly penalised')
print(f'Loss config: Huber(d={HUBER_DELTA}) + {GRAD_WEIGHT}*GradL1 + {FFT_WEIGHT}*FocalFFT(boost={HF_BOOST}x @{HF_CUTOFF}Hz)')


In [ ]:
# CELL 7 — TRAINING ENGINE
#
# Cosine Annealing with Warm Restarts (SGDR) replaces OneCycleLR:
# - T0=20 epochs: first restart at ep 20, then 20, 40, 60...
# - On a periodic signal (ECG), CAWR finds sharper minima than OneCycle
#   because it re-explores from a fresh LR peak after each restart
# - eta_min=1e-6 ensures the LR doesn't kill gradients at cycle trough
#
# Batch size 48 (vs 64 in v12):
# - Smaller batches -> noisier gradient -> better generalisation for
#   periodic signals; also reduces memory pressure on 15-GB GPU
#
# Gradient clipping: 0.5 (tighter than v12's 1.0)
# - BiLSTM can produce large gradients; 0.5 keeps training stable

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total = 0.0
    for xb, yb in tqdm(loader, desc='Train', leave=False, ncols=88):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        total += loss.item() * len(xb)
    scheduler.step()   # CAWR steps once per epoch
    return total / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        pred = model(xb)
        total += criterion(pred, yb).item() * len(xb)
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())
    return (total / len(loader.dataset),
            np.concatenate(preds), np.concatenate(targets))


def train_model(model, tr_loader, vl_loader,
                n_epochs=100, base_lr=2e-4, patience=20):
    global n_params

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=base_lr,
        weight_decay=1e-4, eps=1e-8)

    # CosineAnnealingWarmRestarts: T_0=first restart epoch, T_mult=1 (equal periods)
    scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=1, eta_min=1e-6)

    best_val, no_improve = float('inf'), 0
    ckpt    = os.path.join(CKPT_DIR, 'ECG_CNN_v13_best.pt')
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    sep = '-' * 72
    print(f'\n{sep}')
    print(f'  ECGUNet v13  |  {INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s  |  {n_params:,} params')
    print(f'  Loss: Huber({HUBER_DELTA}) + {GRAD_WEIGHT}*GradL1 + {FFT_WEIGHT}*FocalFFT(hf={HF_BOOST}x)')
    print(f'  CAWR(T0=20) | AdamW(lr={base_lr:.0e}, wd=1e-4) | clip=0.5 | batch={tr_loader.batch_size}')
    print(f'  patience={patience}  n_epochs={n_epochs}')
    print(sep)

    pbar = tqdm(range(1, n_epochs + 1), desc='Epochs', unit='ep', ncols=88)
    for ep in pbar:
        tr_loss       = train_epoch(model, tr_loader, optimizer, scheduler, DEVICE)
        vl_loss, _, _ = eval_epoch(model, vl_loader, DEVICE)
        cur_lr        = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['lr'].append(cur_lr)

        is_best = vl_loss < best_val
        if is_best:
            best_val = vl_loss; no_improve = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_improve += 1

        pbar.set_postfix(tr=f'{tr_loss:.4f}', vl=f'{vl_loss:.4f}',
                         lr=f'{cur_lr:.1e}', pat=no_improve)

        if ep % 5 == 0 or is_best or ep == 1:
            tqdm.write(
                f'  ep {ep:3d}  train={tr_loss:.5f}  val={vl_loss:.5f}'
                f'  lr={cur_lr:.2e}'
                f'{"  * best" if is_best else f"  (no-imp {no_improve}/{patience})"}')

        if no_improve >= patience:
            tqdm.write(f'  Early stop ep {ep}  best val={best_val:.6f}')
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    print(f'\n  Best checkpoint  val={best_val:.6f}  -> {ckpt}')
    print(f'{sep}\n')
    return history

print('OK Training engine ready')


In [ ]:
# CELL 8 — TRAIN
#
# What healthy training should look like vs v12:
#   ep 1 : loss ~2-4 (much lower start — U-Net skips give immediate signal)
#   ep 5 : loss dropping steadily, not plateauing
#   ep 20: first CAWR restart — loss may jump slightly then recover; that's correct
#   ep 40: second restart — should be near best or at best
#   Variance check (Cell 14): pred_std should match true_std on all leads
#   QRS F1 (Cell 13): should be > 0.5
#
# If loss is still > 5.0 after ep 10:
#   -> Check that X and y are truly non-overlapping time windows
#   -> Check NORM_SIGMA is ~1.0 (data was already normalised)
#   -> Raise base_lr to 3e-4 here

history = train_model(model, cnn_tr, cnn_vl,
                      n_epochs=100, base_lr=2e-4, patience=20)


In [ ]:
# CELL 9 — TRAINING HISTORY PLOTS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_range  = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep_range, history['train_loss'], color='#0ea5e9', lw=2,
             label='Train', marker='o', ms=3)
axes[0].plot(ep_range, history['val_loss'], color='#ef4444', lw=2,
             label='Val', marker='s', ms=3, ls='--')
best_ep = int(np.argmin(history['val_loss'])) + 1
axes[0].axvline(best_ep, color='gold', ls=':', lw=2, label=f'Best ep {best_ep}')
# Mark CAWR restart epochs
for r in range(20, len(history['train_loss']), 20):
    axes[0].axvline(r, color='#a78bfa', ls=':', lw=1.2, alpha=0.6)
axes[0].set_title('ECGUNet v13 — Training Loss  (purple dashes = CAWR restarts)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CompositeLoss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, history['lr'], color='#10b981', lw=2)
axes[1].set_title('CosineAnnealingWarmRestarts LR  (T0=20)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning Rate')
axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_ep}  best val: {min(history["val_loss"]):.5f}')
print('OK')


In [ ]:
# CELL 10 — TEST EVALUATION (de-normalised mV)
test_loss, test_preds_n, test_targets_n = eval_epoch(model, cnn_te, DEVICE)
print(f'Test CompositeLoss : {test_loss:.6f}  (normalised)')

# De-normalise to original mV scale
test_preds   = denormalize(test_preds_n)
test_targets = denormalize(test_targets_n)

mae_per_lead, rmse_per_lead, prd_per_lead = [], [], []
for i in range(N_LEADS):
    p, t = test_preds[:,:,i].flatten(), test_targets[:,:,i].flatten()
    mae_per_lead.append(mean_absolute_error(t, p))
    rmse_per_lead.append(np.sqrt(mean_squared_error(t, p)))
    prd = 100.0 * np.sqrt(np.sum((t - p) ** 2) / (np.sum(t ** 2) + 1e-9))
    prd_per_lead.append(prd)

mae_macro  = float(np.mean(mae_per_lead))
rmse_macro = float(np.mean(rmse_per_lead))
prd_macro  = float(np.mean(prd_per_lead))

print(f"\n{'Lead':>6s}   {'MAE(mV)':>8s}   {'RMSE(mV)':>9s}   {'PRD(%)':>7s}")
print('-' * 40)
for i, name in enumerate(LEAD_NAMES):
    status = ('  ✓' if prd_per_lead[i] < 10
              else ('  OK' if prd_per_lead[i] < 25
              else ('  ~' if prd_per_lead[i] < 50 else '  FAIL')))
    print(f'  {name:>4s}   {mae_per_lead[i]:8.4f}   {rmse_per_lead[i]:9.4f}   {prd_per_lead[i]:7.2f}{status}')
print('-' * 40)
print(f'  {"Macro":>4s}   {mae_macro:8.4f}   {rmse_macro:9.4f}   {prd_macro:7.2f}')
print(f'\nPRD: <10% excellent | 10-25% good | 25-50% partial | >50% FAIL')
if prd_macro < 10:
    print('EXCELLENT: ECG morphology well captured!')
elif prd_macro < 25:
    print('GOOD: morphology captured, minor amplitude errors')
elif prd_macro < 50:
    print('PARTIAL: some structure. Try raising HF_BOOST to 8.0 in Cell 6')
else:
    print('FAIL: see Cell 6 comments — verify X/y are non-overlapping windows')
print('OK')


In [ ]:
# CELL 11 — PER-LEAD RMSE + PRD BARS
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_LEADS))

bars = axes[0].bar(np.arange(N_LEADS), rmse_per_lead, width=0.62,
                   color=colors, alpha=0.88, edgecolor='black', lw=0.5)
axes[0].axhline(rmse_macro, color='#facc15', ls='--', lw=2.5,
                label=f'Macro RMSE: {rmse_macro:.4f} mV')
for bar, val in zip(bars, rmse_per_lead):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(np.arange(N_LEADS))
axes[0].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[0].set_ylabel('RMSE (mV)'); axes[0].set_title('Per-Lead RMSE', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

bars2 = axes[1].bar(np.arange(N_LEADS), prd_per_lead, width=0.62,
                    color=colors, alpha=0.88, edgecolor='black', lw=0.5)
axes[1].axhline(prd_macro, color='#facc15', ls='--', lw=2.5,
                label=f'Macro PRD: {prd_macro:.1f}%')
axes[1].axhline(10,  color='green',  ls=':', lw=1.5, label='Excellent (<10%)')
axes[1].axhline(25,  color='orange', ls=':', lw=1.5, label='Good (<25%)')
for bar, val in zip(bars2, prd_per_lead):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(np.arange(N_LEADS))
axes[1].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[1].set_ylabel('PRD (%)'); axes[1].set_title('Per-Lead PRD (clinical)', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_per_lead_rmse_prd.png'), dpi=150, bbox_inches='tight')
plt.show()
print('OK 02_per_lead_rmse_prd.png saved')


In [ ]:
# CELL 12 — PREDICTION OVERLAY (de-normalised mV)
# If the model is working, the red (predicted) line should hug the blue (actual)
# including QRS spikes, P waves, and T waves.
# A flat red line at this stage = the variance check (Cell 13) will explain why.

N_ROWS       = 4
lead_indices = [LEAD_NAMES.index(l) if l in LEAD_NAMES else i
                for i, l in enumerate(['I', 'II', 'V1', 'V5'])]
lead_indices = lead_indices[:4]  # cap at 4
t_axis       = np.arange(HORIZON) / FS

fig, axes = plt.subplots(N_ROWS, 4, figsize=(22, 14))
for row in range(N_ROWS):
    for col, li in enumerate(lead_indices):
        ax     = axes[row, col]
        actual = test_targets[row, :, li]
        pred   = test_preds[row,   :, li]
        rmse_i = np.sqrt(mean_squared_error(actual, pred))
        prd_i  = 100.0 * np.sqrt(np.sum((actual - pred)**2) / (np.sum(actual**2) + 1e-9))
        ax.plot(t_axis, actual, color='#0ea5e9', lw=1.8, label='Actual',    alpha=0.92)
        ax.plot(t_axis, pred,   color='#ef4444', lw=1.4, label='Predicted', alpha=0.88, ls='--')
        ax.set_title(f'{LEAD_NAMES[li]}  RMSE={rmse_i:.3f}mV  PRD={prd_i:.1f}%',
                     fontsize=9, fontweight='bold')
        ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('mV', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.25)
        if col == 0 and row == 0:
            ax.legend(fontsize=8, loc='upper right')

fig.suptitle('ECGUNet v13: Predicted vs Actual  (Blue=Actual | Red=Predicted)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_predictions_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()
print('OK 03_predictions_overlay.png saved')


In [ ]:
# CELL 13 — QRS-DETECTION F1
# A flat prediction -> zero detected peaks -> F1=0 (unfakeable).
# Target: F1 > 0.5. A working model on PTB-XL at 100Hz gets F1 ~0.7-0.85.

TOL_SAMPLES = int(0.05 * FS)   # 50ms tolerance

def detect_peaks_ecg(sig, fs=FS):
    """Simple threshold-based R-peak detector."""
    height = np.mean(sig) + 0.4 * np.std(sig)
    peaks, _ = find_peaks(sig, height=height, distance=int(0.25 * fs))
    return peaks

def qrs_precision_recall_f1(actual_arr, pred_arr, tol=TOL_SAMPLES):
    tp = fp = fn = 0
    for n in range(actual_arr.shape[0]):
        a_peaks = detect_peaks_ecg(actual_arr[n])
        p_peaks = detect_peaks_ecg(pred_arr[n])
        matched = set()
        for pp in p_peaks:
            if len(a_peaks) == 0:
                fp += 1; continue
            dists = np.abs(a_peaks - pp)
            best  = a_peaks[np.argmin(dists)]
            if dists.min() <= tol and best not in matched:
                matched.add(best); tp += 1
            else:
                fp += 1
        fn += len(a_peaks) - len(matched)
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    return prec, rec, f1, tp, fp, fn

# Evaluate on Lead II (most prominent R-peaks)
li    = LEAD_NAMES.index('II') if 'II' in LEAD_NAMES else 1
n_ev  = min(300, test_targets.shape[0])
prec, rec, f1, tp, fp, fn = qrs_precision_recall_f1(
    test_targets[:n_ev, :, li], test_preds[:n_ev, :, li])

print(f'QRS-detection on Lead {LEAD_NAMES[li]}  (n={n_ev}, tol=±50ms)')
print(f'  TP={tp}  FP={fp}  FN={fn}')
print(f'  Precision : {prec:.3f}')
print(f'  Recall    : {rec:.3f}')
print(f'  F1        : {f1:.3f}')
if f1 < 0.3:
    print('FAIL — model not tracking individual heartbeats')
    print('  -> Check PRD: if > 50%, re-verify X/y window construction')
elif f1 < 0.5:
    print('Partial — some beats detected. Try CAWR T0=10 for faster convergence')
else:
    print('Good — model tracking real R-peaks!')


In [ ]:
# CELL 14 — PREDICTION VARIANCE CHECK
# Target: pred_std / true_std > 0.6 on all leads.
# If still <0.4 on all leads after v13: the X/y windows may overlap
# (reconstruction task instead of forecasting task).

pred_std = test_preds.std(axis=0)     # (T, leads)
true_std = test_targets.std(axis=0)   # (T, leads)
t_axis   = np.arange(HORIZON) / FS

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
flat_leads, low_leads = [], []
for i, (ax, name) in enumerate(zip(axes.flatten(), LEAD_NAMES)):
    ax.plot(t_axis, true_std[:,i], color='#0ea5e9', lw=1.4, label='Actual std')
    ax.plot(t_axis, pred_std[:,i], color='#ef4444', lw=1.4, label='Pred std', ls='--')
    ratio = pred_std[:,i].mean() / (true_std[:,i].mean() + 1e-9)
    ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('Std (mV)', fontsize=8)
    ax.grid(alpha=0.25)
    if i == 0: ax.legend(fontsize=8)
    if ratio < 0.3:
        ax.set_facecolor('#fff0f0')
        ax.set_title(f'Lead {name} [{ratio:.2f}x] FLAT', color='red', fontweight='bold', fontsize=9)
        flat_leads.append(name)
    elif ratio < 0.6:
        ax.set_facecolor('#fffbe6')
        ax.set_title(f'Lead {name} [{ratio:.2f}x] low', color='darkorange', fontweight='bold', fontsize=9)
        low_leads.append(name)
    else:
        ax.set_title(f'Lead {name} [{ratio:.2f}x] OK', fontweight='bold', fontsize=9)

fig.suptitle('ECGUNet v13 — Prediction Variance Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_variance_check.png'), dpi=150, bbox_inches='tight')
plt.show()

if flat_leads and len(flat_leads) == N_LEADS:
    print('ALL LEADS FLAT: model outputting near-constant predictions.')
    print('  -> Verify that X (past) and y (future) are truly non-overlapping windows.')
    print('  -> Print X_train[0,:5,0] and y_train[0,:5,0] to check.')
elif flat_leads:
    print(f'Flat leads: {flat_leads}  -> Raise HF_BOOST to 8.0 in Cell 6')
elif low_leads:
    print(f'Low-variance leads: {low_leads}  -> train longer or raise GRAD_WEIGHT')
else:
    ov = pred_std.mean() / true_std.mean()
    print(f'OK Variance tracking. Overall ratio = {ov:.2f}x')


In [ ]:
# CELL 15 — SAVE RESULTS
results = dict(
    model='ECGUNet_v13_CNN_BiLSTM_UNet',
    horizon_s=HORIZON/FS, input_s=INPUT_LEN/FS,
    n_parameters=n_params,
    test_loss=float(test_loss),
    mae_per_lead=mae_per_lead, mae_macro=mae_macro,
    rmse_per_lead=rmse_per_lead, rmse_macro=rmse_macro,
    prd_per_lead=prd_per_lead, prd_macro=prd_macro,
    qrs_precision=float(prec), qrs_recall=float(rec), qrs_f1=float(f1),
    loss_config=dict(huber_delta=HUBER_DELTA, grad_weight=GRAD_WEIGHT,
                     fft_weight=FFT_WEIGHT, hf_boost=HF_BOOST, hf_cutoff=HF_CUTOFF),
    norm=dict(mu=NORM_MU, sigma=NORM_SIGMA),
    history=history, lead_names=LEAD_NAMES,
    test_preds=test_preds, test_targets=test_targets,
)
res_path = os.path.join(CKPT_DIR, 'ECG_CNN_v13_results.pkl')
with open(res_path, 'wb') as f:
    pickle.dump(results, f)

print(f'\n{"="*64}')
print(f'  ECGUNet v13 FINAL  ({INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s)')
print(f'{"="*64}')
print(f'  Architecture  : CNN-BiLSTM U-Net (skip connections)')
print(f'  Parameters    : {n_params:,}')
print(f'  Macro MAE     : {mae_macro:.6f} mV')
print(f'  Macro RMSE    : {rmse_macro:.6f} mV')
print(f'  Macro PRD     : {prd_macro:.2f}%   (<10% excellent, <25% good)')
print(f'  QRS F1        : {f1:.3f}  (P={prec:.3f} R={rec:.3f})')
print(f'{"="*64}')
print(f'  Checkpoint : {CKPT_DIR}/ECG_CNN_v13_best.pt')
print(f'  Results    : {res_path}')
print('OK v13 complete')
